### Clustering Problem

**PROBLEM: Customer Segmentation**

Using the article [here](https://www.semanticscholar.org/paper/Data-mining-for-the-online-retail-industry%3A-A-case-Chen-Sain/e43a5a90fa33d419df42e485099f8f08badf2149) and associated dataset loaded below, your goal is to apply unsupervised learning to a problem in the retail industry.  Your goal is to:

1. Build appropriate features and perform clustering based on RFM.
2. Provide insight into who makes up your clusters and how this information could be used advantageously by the business.

You are to present your key findings to another group during class, and will in turn comment and give feedback on the second groups findings.

Timeline:

1. ~30 minutes to build features and cluster data
2. ~30 minutes to analyze results and highlight key findings
3. ~20 minutes to share findings with a second group
4. ~10 minutes reflection on findings and other groups work

In [4]:
# pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo 

In [3]:
online_retail = fetch_ucirepo(id=352) 
X = online_retail.data.features
X.head()

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [6]:
import pandas as pd

In [7]:
X['InvoiceDate'] = pd.to_datetime(X['InvoiceDate'])

/var/folders/8v/7bhy8yqn04b7rzqglb2s38200000gn/T/ipykernel_19236/214597014.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['InvoiceDate'] = pd.to_datetime(X['InvoiceDate'])


In [9]:
rfm = X.groupby('CustomerID')[['InvoiceDate']].max()

In [12]:
max_date = rfm['InvoiceDate'].max()

In [15]:
rfm['recency'] = (max_date - rfm['InvoiceDate']).dt.days

In [18]:
rfm.drop('InvoiceDate', axis = 1, inplace = True)

In [19]:
rfm.head()

,recency
CustomerID,
12346.0,325
12347.0,1
12348.0,74
12349.0,18
12350.0,309


In [22]:
rfm['frequency'] = X.groupby('CustomerID')['InvoiceDate'].count()

In [24]:
X['total'] = X['Quantity'] * X['UnitPrice']

/var/folders/8v/7bhy8yqn04b7rzqglb2s38200000gn/T/ipykernel_19236/3746215797.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['total'] = X['Quantity'] * X['UnitPrice']


In [26]:
rfm['monetary'] = X.groupby('CustomerID')['total'].sum()

In [27]:
rfm.head()

,recency,frequency,monetary
CustomerID,,,
12346.0,325,2,0.00
12347.0,1,182,4310.00
12348.0,74,31,1797.24
12349.0,18,73,1757.55
12350.0,309,17,334.40


In [29]:
from sklearn.preprocessing import StandardScaler

In [31]:
scaler = StandardScaler()
X = scaler.fit_transform(rfm)

In [32]:
X

array([[ 2.32202285, -0.3917197 , -0.23100099],
       [-0.89373323,  0.38265697,  0.29343167],
       [-0.1691956 , -0.26695902, -0.01231622],
       ...,
       [-0.83418219, -0.34439668, -0.20951263],
       [-0.87388289,  2.85205812,  0.02390005],
       [-0.48680114, -0.0991774 , -0.00744423]])

In [33]:
from sklearn.cluster import KMeans

In [34]:
kmeans = KMeans(5, random_state = 42).fit(X)

In [35]:
rfm['cluster'] = kmeans.labels_

In [36]:
rfm.head()

,recency,frequency,monetary,cluster
CustomerID,,,,
12346.0,325,2,0.00,2
12347.0,1,182,4310.00,0
12348.0,74,31,1797.24,0
12349.0,18,73,1757.55,0
12350.0,309,17,334.40,2


In [37]:
rfm.groupby('cluster').mean()

,recency,frequency,monetary
cluster,,,
0,41.616103,79.494686,1364.023988
1,2.666667,956.333333,241136.560000
2,248.382682,27.624767,464.988576
3,1.000000,5914.000000,64776.602500
4,11.102151,558.086022,13886.330699


In [28]:
(rfm - rfm.mean()) / rfm.std()

,recency,frequency,monetary
CustomerID,,,
12346.0,2.321757,-0.391675,-0.230975
12347.0,-0.893631,0.382613,0.293398
12348.0,-0.169176,-0.266928,-0.012315
12349.0,-0.724922,-0.086261,-0.017144
12350.0,2.162973,-0.327151,-0.190290
...,...,...,...
18280.0,1.845403,-0.357262,-0.209002
18281.0,0.882772,-0.370167,-0.221142
18282.0,-0.834087,-0.344357,-0.209489
